# OSL-Words Dataset Re-Splitting (Signer-Aware)
Splits the **original (non-augmented)** OSL-Words videos into train/dev/test using **signer-aware** stratification.

**Key principle:** All takes from the same signer for the same word are an **indivisible unit** — they always go to the same split.

**Rules (based on unique signers per word):**
- Words with **1 signer** → all samples to train only
- Words with **2 signers** → 1 signer group to train, 1 signer group to dev (excluded from test)
- Words with **≥3 signers** → signer groups distributed across train/dev/test

Output saved to `reports/new_split/` — **does NOT modify existing label files**.

In [8]:
import gzip, pickle, re, json, random
import pandas as pd
from collections import defaultdict
from pathlib import Path
import math

SEED = 42
random.seed(SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────
UNISIGN_DATA = Path(r"C:\Users\MOBPC\Downloads\FYP\FYPproject\Uni-Sign-main\Uni-Sign-main\data\OSL-Words")
OUT_DIR      = Path(r"C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\notebooks\reports\new_split")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_labels(path):
    with gzip.open(path, 'rb') as f:
        return pickle.load(f)

train_labels = load_labels(UNISIGN_DATA / 'labels-osl.train')
dev_labels   = load_labels(UNISIGN_DATA / 'labels-osl.dev')
test_labels  = load_labels(UNISIGN_DATA / 'labels-osl.test')

print(f"Loaded train (augmented): {len(train_labels)} entries")
print(f"Loaded dev  (original)  : {len(dev_labels)} entries")
print(f"Loaded test (original)  : {len(test_labels)} entries")

Loaded train (augmented): 23368 entries
Loaded dev  (original)  : 185 entries
Loaded test (original)  : 34 entries


In [9]:
# ── Step 1: Reconstruct full original dataset ──────────────────────────────────
originals = {}  # base_id -> {name, text, video_path, signer_id}

# Extract originals from augmented train filenames
for k, v in train_labels.items():
    m = re.search(r'(\d{4}_S\d+_T\d+)$', k)
    if m:
        base_id = m.group(1)
        if base_id not in originals:
            signer_m = re.search(r'_(S\d+)_', base_id)
            originals[base_id] = {
                'name':       base_id,
                'text':       v['text'],
                'video_path': base_id + '.mp4',
                'signer_id':  signer_m.group(1) if signer_m else 'UNKNOWN',
            }

# Add dev and test originals
for k, v in {**dev_labels, **test_labels}.items():
    signer_m = re.search(r'_(S\d+)_', k)
    originals[k] = {
        'name':       k,
        'text':       v['text'],
        'video_path': v['video_path'],
        'signer_id':  signer_m.group(1) if signer_m else 'UNKNOWN',
    }

print(f"Total original videos reconstructed: {len(originals)}")
print(f"Unique words/classes               : {len(set(v['text'] for v in originals.values()))}")

# ── Step 2: Group by word → signer ────────────────────────────────────────────
# word_to_signers[word][signer_id] = [list of base_ids]
word_to_signers = defaultdict(lambda: defaultdict(list))
word_to_videos  = defaultdict(list)  # kept for backward compat

for vid_id, info in originals.items():
    word_to_signers[info['text']][info['signer_id']].append(vid_id)
    word_to_videos[info['text']].append(vid_id)

# Sort videos within each signer group for reproducibility
for word in word_to_signers:
    for signer in word_to_signers[word]:
        word_to_signers[word][signer] = sorted(word_to_signers[word][signer])

print(f"\nUnique signer count per word distribution:")
signer_counts = Counter(len(signers) for signers in word_to_signers.values())
for c in sorted(signer_counts):
    print(f"  {c} signer(s): {signer_counts[c]} words")

Total original videos reconstructed: 1235
Unique words/classes               : 692

Unique signer count per word distribution:
  1 signer(s): 549 words
  2 signer(s): 115 words
  3 signer(s): 19 words
  4 signer(s): 9 words


In [10]:
# ── Step 3: Apply signer-aware split logic ─────────────────────────────────────
split_train = {}
split_dev   = {}
split_test  = {}

word_split_log   = {}  # word -> {video_id: split}
signer_split_log = {}  # word -> {signer_id: split}

stats = {'train_only_1': 0, 'train_dev_2': 0, 'full_split_3p': 0}

for word, signers_dict in word_to_signers.items():
    # Shuffle signer list with fixed seed for reproducibility
    signer_ids = sorted(signers_dict.keys())
    random.shuffle(signer_ids)

    n_signers = len(signer_ids)
    word_split_log[word]   = {}
    signer_split_log[word] = {}

    def assign_signer(signer_id, target_split, target_dict):
        for vid_id in signers_dict[signer_id]:
            target_dict[vid_id] = originals[vid_id]
            word_split_log[word][vid_id] = target_split
        signer_split_log[word][signer_id] = target_split

    if n_signers == 1:
        # 1 signer → all videos go to train
        assign_signer(signer_ids[0], 'train', split_train)
        stats['train_only_1'] += 1

    elif n_signers == 2:
        # 2 signers → signer[0] train, signer[1] dev
        assign_signer(signer_ids[0], 'train', split_train)
        assign_signer(signer_ids[1], 'dev',   split_dev)
        stats['train_dev_2'] += 1

    else:
        # 3+ signers → distribute signer groups across train/dev/test
        n_test  = max(1, math.floor(n_signers * 0.15))
        n_dev   = max(1, math.floor(n_signers * 0.15))
        n_train = n_signers - n_dev - n_test
        if n_train < 1:
            n_train = 1
            n_dev = max(1, n_signers - n_train - n_test)
            n_test = n_signers - n_train - n_dev

        train_signers = signer_ids[:n_train]
        dev_signers   = signer_ids[n_train:n_train + n_dev]
        test_signers  = signer_ids[n_train + n_dev:]

        for s in train_signers:
            assign_signer(s, 'train', split_train)
        for s in dev_signers:
            assign_signer(s, 'dev', split_dev)
        for s in test_signers:
            assign_signer(s, 'test', split_test)
        stats['full_split_3p'] += 1

print("Split complete.")
print(f"  Words with 1 signer  (train only)         : {stats['train_only_1']}")
print(f"  Words with 2 signers (train+dev, no test)  : {stats['train_dev_2']}")
print(f"  Words with 3+ signers (full split)         : {stats['full_split_3p']}")
print()
print(f"Train originals : {len(split_train)}")
print(f"Dev originals   : {len(split_dev)}")
print(f"Test originals  : {len(split_test)}")
print(f"Total           : {len(split_train)+len(split_dev)+len(split_test)} (should be {len(originals)})")

Split complete.
  Words with 1 signer  (train only)         : 549
  Words with 2 signers (train+dev, no test)  : 115
  Words with 3+ signers (full split)         : 28

Train originals : 943
Dev originals   : 236
Test originals  : 56
Total           : 1235 (should be 1235)


In [12]:
# ── Step 4: Validate the split ──────────────────────────────────────────────────
errors = []

# 1. No overlap between splits
train_ids = set(split_train.keys())
dev_ids   = set(split_dev.keys())
test_ids  = set(split_test.keys())

overlap_td = train_ids & dev_ids
overlap_tt = train_ids & test_ids
overlap_dt = dev_ids & test_ids

if overlap_td:
    errors.append(f"Train ∩ Dev overlap: {len(overlap_td)} ids")
if overlap_tt:
    errors.append(f"Train ∩ Test overlap: {len(overlap_tt)} ids")
if overlap_dt:
    errors.append(f"Dev ∩ Test overlap: {len(overlap_dt)} ids")

# 2. Completeness
total = len(train_ids) + len(dev_ids) + len(test_ids)
if total != len(originals):
    errors.append(f"Total {total} ≠ originals {len(originals)}")

# 3. Signer cross-split check: no signer in >1 split for the same word
signer_leak = []
for word, s_log in signer_split_log.items():
    signer_to_split = {}
    for signer_id, split_name in s_log.items():
        if signer_id in signer_to_split and signer_to_split[signer_id] != split_name:
            signer_leak.append((word, signer_id, signer_to_split[signer_id], split_name))
        signer_to_split[signer_id] = split_name
if signer_leak:
    errors.append(f"Signer leaks across splits: {len(signer_leak)} cases")
    for w, s, s1, s2 in signer_leak[:5]:
        errors.append(f"  word={w}, signer={s}: {s1} vs {s2}")

# 4. Words in test have ≥3 signers
test_words = set()
for vid_id, info in split_test.items():
    test_words.add(info['text'])
test_words_with_few_signers = [w for w in test_words if len(word_to_signers[w]) < 3]
if test_words_with_few_signers:
    errors.append(f"Test words with < 3 signers: {test_words_with_few_signers}")

if errors:
    print("VALIDATION FAILED:")
    for e in errors:
        print(f"  ✗ {e}")
else:
    print("✓ All validations passed.")
    print(f"  • No overlaps between splits")
    print(f"  • All {len(originals)} originals accounted for")
    print(f"  • No signer leaks across splits")
    print(f"  • All test words have ≥ 3 signers")

✓ All validations passed.
  • No overlaps between splits
  • All 1235 originals accounted for
  • No signer leaks across splits
  • All test words have ≥ 3 signers


In [5]:
# ── Step 5: Save plain-text label files (video_name label) ─────────────────────

def save_txt_labels(split_dict, path):
    with open(path, 'w', encoding='utf-8') as f:
        for vid_id, info in sorted(split_dict.items()):
            f.write(f"{vid_id} {info['text']}\n")
    print(f"Saved: {path} ({len(split_dict)} entries)")

save_txt_labels(split_train, OUT_DIR / 'labels-osl.train.txt')
save_txt_labels(split_dev,   OUT_DIR / 'labels-osl.dev.txt')
save_txt_labels(split_test,  OUT_DIR / 'labels-osl.test.txt')

Saved: C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\notebooks\reports\new_split\labels-osl.train.txt (817 entries)
Saved: C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\notebooks\reports\new_split\labels-osl.dev.txt (288 entries)
Saved: C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\notebooks\reports\new_split\labels-osl.test.txt (130 entries)


In [6]:
# ── Step 6: Save gzip-pickle label files (same format as training pipeline) ────
# Format: {video_name: {'name': ..., 'text': ..., 'video_path': ...}}

def save_gzip_labels(split_dict, path):
    data = {}
    for vid_id, info in split_dict.items():
        data[vid_id] = {
            'name':       info['name'],
            'text':       info['text'],
            'video_path': info['video_path'],
        }
    with gzip.open(path, 'wb') as f:
        pickle.dump(data, f)
    print(f"Saved: {path} ({len(data)} entries)")

save_gzip_labels(split_train, OUT_DIR / 'labels-osl.train')
save_gzip_labels(split_dev,   OUT_DIR / 'labels-osl.dev')
save_gzip_labels(split_test,  OUT_DIR / 'labels-osl.test')
print()
print("NOTE: These are ORIGINAL-only splits.")
print("To use for training, augment the train split first, then replace the files in:")
print(f"  {UNISIGN_DATA}")

Saved: C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\notebooks\reports\new_split\labels-osl.train (817 entries)
Saved: C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\notebooks\reports\new_split\labels-osl.dev (288 entries)
Saved: C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\notebooks\reports\new_split\labels-osl.test (130 entries)

NOTE: These are ORIGINAL-only splits.
To use for training, augment the train split first, then replace the files in:
  C:\Users\MOBPC\Downloads\FYP\FYPproject\Uni-Sign-main\Uni-Sign-main\data\OSL-Words


In [ ]:
# ── Step 7: Save JSON log ───────────────────────────────────────────────────────
split_log = {
    'seed': SEED,
    'total_originals': len(originals),
    'total_words': len(word_to_signers),
    'stats': stats,
    'counts': {
        'train': len(split_train),
        'dev':   len(split_dev),
        'test':  len(split_test),
    },
    'word_split_log':   word_split_log,
    'signer_split_log': signer_split_log,
}

json_path = OUT_DIR / 'resplit_log.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(split_log, f, ensure_ascii=False, indent=2)
print(f"Saved: {json_path}")

# Preview signer assignments for a few words
print("\n── Sample signer→split mapping ──")
for i, (word, s_log) in enumerate(signer_split_log.items()):
    if i >= 5:
        break
    n_s = len(s_log)
    assignments = ', '.join(f'{s}→{sp}' for s, sp in s_log.items())
    print(f"  '{word}' ({n_s} signers): {assignments}")

Saved split log → C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\notebooks\reports\new_split\word_split_assignment.json

Word: 'حساسية'
  0019_S02_T02                   → train
  0019_S01_T02                   → train
  0019_S02_T03                   → dev
  0019_S01_T01                   → test

Word: 'طفل'
  0096_S01_T02                   → train
  0096_S07_T01                   → dev
  0096_S01_T01                   → test

Word: 'مستعمل'
  0200_S03_T02                   → train
  0200_S03_T01                   → dev
  0200_S07_T01                   → test



In [ ]:
# ── Step 8: Print summary ───────────────────────────────────────────────────────
train_words = set(info['text'] for info in split_train.values())
dev_words   = set(info['text'] for info in split_dev.values())
test_words  = set(info['text'] for info in split_test.values())

summary = f"""╔══════════════════════════════════════════════════════════════╗
║            OSL Signer-Aware Re-Split Summary                ║
╠══════════════════════════════════════════════════════════════╣
║  Total originals     : {len(originals):>6}                              ║
║  Unique words        : {len(word_to_signers):>6}                              ║
║  Random seed         : {SEED:>6}                              ║
╠══════════════════════════════════════════════════════════════╣
║                       Videos   Words                        ║
║  Train             : {len(split_train):>6}   {len(train_words):>5}                        ║
║  Dev               : {len(split_dev):>6}   {len(dev_words):>5}                        ║
║  Test              : {len(split_test):>6}   {len(test_words):>5}                        ║
╠══════════════════════════════════════════════════════════════╣
║  Words 1 signer  (train only)       : {stats['train_only_1']:>5}                  ║
║  Words 2 signers (train+dev)        : {stats['train_dev_2']:>5}                  ║
║  Words 3+ signers (full split)      : {stats['full_split_3p']:>5}                  ║
╠══════════════════════════════════════════════════════════════╣
║  Signer leakage validation : {'PASS' if not signer_leak else 'FAIL':>6}                        ║
╚══════════════════════════════════════════════════════════════╝"""

print(summary)

# Save summary text
summary_path = OUT_DIR / 'resplit_summary.txt'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(summary)
print(f"\nSaved: {summary_path}")

In [ ]:
# ── Step 9: Per-class test table ────────────────────────────────────────────────
if split_test:
    rows = []
    for vid_id, info in split_test.items():
        m = re.search(r'_(S\d+)_', vid_id)
        signer = m.group(1) if m else 'unknown'
        rows.append({'video_id': vid_id, 'word': info['text'], 'signer': signer})

    df_test = pd.DataFrame(rows).sort_values(['word', 'signer', 'video_id']).reset_index(drop=True)

    # Count signers and videos per test word
    test_summary = (
        df_test.groupby('word')
        .agg(
            n_videos=('video_id', 'count'),
            n_signers=('signer', 'nunique'),
            signers=('signer', lambda x: ', '.join(sorted(x.unique()))),
            total_signers_for_word=('word', lambda x: len(word_to_signers[x.iloc[0]])),
        )
        .sort_values('word')
    )
    test_summary.columns = ['test_videos', 'test_signers', 'test_signer_ids', 'total_signers']
    
    csv_path = OUT_DIR / 'resplit_test_detail.csv'
    df_test.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")

    print(f"\nTest set: {len(df_test)} videos, {df_test['word'].nunique()} words, "
          f"{df_test['signer'].nunique()} unique signers\n")
    print(test_summary.to_string())
else:
    print("Test set is empty — no words with ≥ 3 signers.")